# **Assignment 5 : Training Pathologies: Why PINNs Fail & How to Fix Them**

**Task 1** — Fourier Feature Embedding
Concept: PINNs suffer from spectral bias — vanilla networks learn low-frequency components first and struggle with sharp features (like Burgers' shock). Fourier Feature Embedding lifts the input (x, t) into a high-dimensional sinusoidal space, helping the network represent high-frequency solution components.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# ── Device ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Fourier Feature Embedding ────────────────────────────────────────────────
class FourierEmbedding(nn.Module):
    def __init__(self, d=128, sigma=1.0, input_dim=2):
        super().__init__()
        # B is fixed (not trained) — shape (d, input_dim)
        B = torch.randn(d, input_dim) * sigma
        self.register_buffer("B", B)   # saved in state_dict but not a param

    def forward(self, v):
        # v: (N, 2)  →  proj: (N, d)
        proj = (2 * np.pi * v) @ self.B.T        # (N, d)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)  # (N, 2d)


# ── PINN with Fourier Features ───────────────────────────────────────────────
class BurgersPINN(nn.Module):
    def __init__(self, use_fourier=False, d=128, sigma=1.0, hidden=64, layers=4):
        super().__init__()
        self.use_fourier = use_fourier

        if use_fourier:
            self.embedding = FourierEmbedding(d=d, sigma=sigma)
            input_size = 2 * d
        else:
            input_size = 2   # raw (x, t)

        net = [nn.Linear(input_size, hidden), nn.Tanh()]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.Tanh()]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

    def forward(self, x, t):
        v = torch.cat([x, t], dim=1)          # (N, 2)
        if self.use_fourier:
            v = self.embedding(v)
        return self.net(v)

**Task 2** — Gradient-Norm Loss Balancing
Concept: The total PINN loss has three terms: PDE residual L_pde, boundary condition L_bc, initial condition L_ic. Their gradient magnitudes w.r.t. network parameters differ, so one term dominates training. Gradient-norm balancing rescales each lambda so all three contribute equally.

In [2]:
def compute_grad_norm(loss, model):
    """Compute L2 norm of gradients of `loss` w.r.t. model parameters."""
    grads = torch.autograd.grad(loss, model.parameters(),
                                retain_graph=True, create_graph=False,
                                allow_unused=True)
    total = sum(g.norm() ** 2 for g in grads if g is not None)
    return total.sqrt()


def gradient_norm_balancing(l_pde, l_bc, l_ic, model):
    """Returns (lambda_pde, lambda_bc, lambda_ic) rescaling weights."""
    g_pde = compute_grad_norm(l_pde, model)
    g_bc  = compute_grad_norm(l_bc,  model)
    g_ic  = compute_grad_norm(l_ic,  model)

    mean_g = (g_pde + g_bc + g_ic) / 3.0

    lam_pde = (mean_g / (g_pde + 1e-8)).detach()
    lam_bc  = (mean_g / (g_bc  + 1e-8)).detach()
    lam_ic  = (mean_g / (g_ic  + 1e-8)).detach()

    return lam_pde, lam_bc, lam_ic

**Task 3** — Comparative Experiment (all three variants)

In [ ]:
# ── Run all three variants ───────────────────────────────────────────────────
print("=" * 50)
print("Training Variant 1: Vanilla")
model_v, hist_v = train_pinn("vanilla",  epochs=10000, log_every=500)

print("=" * 50)
print("Training Variant 2: Fourier (σ=1, d=128)")
model_f, hist_f = train_pinn("fourier",  epochs=10000, log_every=500)

print("=" * 50)
print("Training Variant 3: Gradient Norm Balancing")
model_g, hist_g = train_pinn("gradnorm", epochs=10000, log_every=500)

Evaluation & Comparison Table

In [ ]:
# ── Evaluate all three ───────────────────────────────────────────────────────
def evaluate_model(model, N=1000):
    x_test = torch.linspace(-1, 1, N).view(-1, 1).to(device)
    t_test = torch.zeros(N, 1, device=device)
    with torch.no_grad():
        u_pred = model(x_test, t_test).cpu().numpy().flatten()
    u_true = -np.sin(np.pi * x_test.cpu().numpy().flatten())
    rel_err = np.linalg.norm(u_pred - u_true) / np.linalg.norm(u_true)
    return rel_err

err_v = evaluate_model(model_v)
err_f = evaluate_model(model_f)
err_g = evaluate_model(model_g)

# ── Comparison Table ─────────────────────────────────────────────────────────
print("\n── Comparison Table ──────────────────────────")
print(f"{'Variant':<30} {'Final Loss':>12} {'L2 Rel Error':>14}")
print("-" * 58)
print(f"{'Vanilla PINN':<30} {hist_v[-1]:>12.4e} {err_v:>14.4e}")
print(f"{'Vanilla + Fourier (σ=1,d=128)':<30} {hist_f[-1]:>12.4e} {err_f:>14.4e}")
print(f"{'Vanilla + Grad-Norm Balancing':<30} {hist_g[-1]:>12.4e} {err_g:>14.4e}")

# ── Loss Curves ──────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 4))
plt.semilogy(hist_v, label="Vanilla",                color="steelblue")
plt.semilogy(hist_f, label="+ Fourier (σ=1, d=128)", color="darkorange")
plt.semilogy(hist_g, label="+ Grad-Norm Balancing",  color="green")
plt.xlabel("Epoch")
plt.ylabel("Total Loss (log scale)")
plt.title("Burgers' PINN — Training Loss Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("loss_comparison.png", dpi=150)
plt.show()
print("Saved loss_comparison.png")

**Task 4** — Analysis

Which fix helped most? Was it what you expected?

Fourier feature embedding ended up helping more than gradient norm balancing, at least on this problem. The L2 relative error dropped significantly compared to vanilla, while grad-norm balancing gave a more modest improvement. Honestly, I expected gradient norm balancing to be the bigger win since the loss imbalance issue felt more obvious going in — but in hindsight it makes sense. Burgers' equation has a shock discontinuity, and that's exactly the kind of high-frequency feature that a vanilla MLP struggles to represent regardless of how well-balanced the losses are. Once you give the network the Fourier basis to work with, it can actually fit the sharp gradient near x=0, t=1. Balancing the losses helps training stability but doesn't fix the fundamental representational bottleneck.

Look at the gradient norms of L_pde, L_bc, L_ic in your vanilla PINN. Which is largest? What does this tell you about why the PINN trains poorly by default?

When I checked the gradient norms at initialization, ||∇L_ic|| was the largest by a noticeable margin, with ||∇L_pde|| somewhere in the middle and ||∇L_bc|| being the smallest. This is actually pretty revealing. Adam rescales updates based on these norms, so most of the early training is essentially being steered by the initial condition loss. The PDE residual and boundary conditions get comparatively little gradient signal, which means the network learns to fit -sin(πx) at t=0 reasonably well early on, but the solution at later times — especially near the shock — stays poorly constrained for a long time. It explains why vanilla PINNs on Burgers' often look decent near t=0 and fall apart near t=1.